# Robô de triagem da Cobratec — modelo rodando no Colab

Sobe um **Ollama com GPU** aqui no Colab e o entrega ao inventário por um túnel
autenticado. Serve para a máquina do escritório não ter que segurar o modelo.

```
WhatsApp → WAHA → inventário (/chat) → túnel → [Colab] proxy → Ollama (GPU)
```

## Leia antes de rodar

**Isto é o caminho de TESTE.** A decisão 31 escolheu rodar o modelo dentro da
empresa porque a fala do devedor não deve sair dela. Apontar o inventário para
cá manda a mensagem do devedor para uma VM do Google — o que é aceitável para
**medir modelo e afinar a triagem com mensagens inventadas**, e não é aceitável
com devedor real. A tela `/chat → Conexão` avisa em vermelho quando o modelo
está fora da rede; se o aviso estiver aparecendo em produção, algo está errado.

O que o robô faz continua o mesmo, aqui ou lá: ele **não fala de valor, acordo
nem pagamento** — isso é barrado por código no inventário (`lib/chat-bot.ts`),
não pelo modelo. Trocar de máquina não afrouxa nada disso.

**Limites do Colab, que você vai encontrar:** a sessão cai sozinha depois de
~90 min sem uso e tem teto de ~12h; o endereço do túnel **muda a cada vez** que
você roda de novo (e o `.env` do inventário precisa ser atualizado); e servir
tráfego contínuo não é o uso que o Colab se propõe a suportar. Para valer,
o modelo mora numa máquina sua.

## Como usar

**O sentido é daqui para o seu servidor.** Este notebook não precisa de nada do
seu `.env` — nenhuma senha, nenhum endereço, nada. Ele *gera* três linhas, e
você as leva para a máquina onde o inventário roda:

```
[Colab]  célula 4 imprime  ──►  você copia  ──►  .env do inventário  ──►  docker compose up -d
```

1. **Ambiente de execução → Alterar o tipo → GPU (T4)**. Sem GPU não vale a
   pena: a CPU do Colab é mais lenta que a do escritório.
2. Rode as células **1 a 4** em ordem.
3. A célula 4 termina imprimindo `OLLAMA_URL`, `OLLAMA_MODELO` e `OLLAMA_TOKEN`.
   Copie as três **para o `.env` do inventário**, substituindo o que houver, e
   recrie o app com `docker compose up -d` (é o que faz ele reler o arquivo).

   > **Se o código do inventário mudou desde o último build, use
   > `docker compose up -d --build`.** Sem o `--build`, o compose recria o
   > container mas reaproveita a imagem antiga: as variáveis novas entram e o
   > código que as usa não. O sintoma é o robô falhar com **401** enquanto o
   > mesmo token funciona no `curl` — foi assim que este aviso nasceu.
4. Confira em **/chat → Conexão**: o card "Quem fala primeiro com o devedor"
   passa a mostrar o aviso vermelho de *modelo fora da rede*. Aparecendo o
   aviso, ligou certo.
5. Deixe a célula 5 rodando e **a aba aberta** — é o que segura a sessão viva.

Para voltar ao modelo de casa: restaure `OLLAMA_URL` para o endereço local,
apague `OLLAMA_TOKEN` e recrie o app. Apagando `OLLAMA_URL` inteiro, o robô sai
de cena e todo atendimento volta para a fila da operadora.

## 1. GPU e instalação do Ollama

In [ ]:
import shutil, subprocess

# Sem GPU o Colab não ajuda em nada: a CPU dele é mais fraca que a de um
# desktop de escritório. Melhor descobrir agora que depois de baixar 2GB.
#
# `shutil.which` e não `subprocess.run` direto: numa sessão sem GPU o
# nvidia-smi NÃO EXISTE, e chamá-lo levanta FileNotFoundError em vez de
# devolver código de erro. Procurar o binário antes evita transformar
# "faltou escolher a GPU" num traceback que parece defeito do notebook.
if not shutil.which("nvidia-smi"):
    raise SystemExit(
        "SEM GPU nesta sessão.\n"
        "Ambiente de execução → Alterar o tipo de ambiente → T4 GPU → Salvar,\n"
        "e rode esta célula de novo. Em CPU o Colab é mais lento que a máquina\n"
        "do escritório, então não vale a pena continuar."
    )

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode != 0:
    raise SystemExit(f"nvidia-smi respondeu erro: {gpu.stderr.strip()}")
print("GPU:", gpu.stdout.strip())


def instalado():
    return shutil.which("ollama") is not None


# O pacote do Ollama vem comprimido em zstd, e a imagem do Colab NÃO traz o
# descompressor. Sem ele o instalador oficial para e imprime "instale o zstd" —
# que foi exatamente onde esta célula morreu antes. Vem primeiro porque os dois
# caminhos de instalação dependem dele.
if not shutil.which("unzstd"):
    print("Instalando o zstd (o Colab não traz, e o pacote do Ollama precisa)…")
    if subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"]).returncode != 0:
        subprocess.run(["apt-get", "-qq", "update"])
        subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"])
    if not shutil.which("unzstd"):
        raise SystemExit("não consegui instalar o zstd — rode a célula de novo")

# Instalação por dois caminhos, e conferindo o resultado em vez do código de
# saída. Isto NÃO é excesso de zelo: o script oficial baixa ~1,4 GB de um CDN
# que já resetou a conexão no meio (`curl: (56)`) aqui no Colab. Pior, a falha é
# silenciosa quando se usa `!curl ... | sh`: o magic falha, o Python segue, e o
# erro só aparece na célula seguinte como "ollama não encontrado".
if not instalado():
    print("Instalando pelo script oficial…")
    subprocess.run(
        "curl -fsSL --retry 3 --retry-all-errors https://ollama.com/install.sh | sh",
        shell=True,
    )

if not instalado():
    # Mesmo pacote, outro caminho: o release do GitHub costuma passar quando o
    # CDN do ollama.com cai. `--retry-all-errors -C -` retoma de onde parou em
    # vez de recomeçar 1,4 GB.
    print("O script oficial não concluiu. Baixando o pacote do GitHub…")
    subprocess.run(
        ["curl", "-fL", "--retry", "5", "--retry-delay", "3", "--retry-all-errors",
         "-C", "-", "-o", "/tmp/ollama.tar.zst",
         "https://github.com/ollama/ollama/releases/latest/download/"
         "ollama-linux-amd64.tar.zst"],
        check=True,
    )
    # O pacote traz `bin/ollama` na raiz: extraído aqui, cai em /usr/local/bin,
    # que já está no PATH.
    subprocess.run(
        ["tar", "--use-compress-program=unzstd", "-xf", "/tmp/ollama.tar.zst",
         "-C", "/usr/local"],
        check=True,
    )

if not instalado():
    raise SystemExit(
        "não consegui instalar o Ollama pelos dois caminhos.\n"
        "Quase sempre é rede do Colab: espere um minuto e rode a célula de novo."
    )
print("Ollama instalado em", shutil.which("ollama"))

## 2. Subir o Ollama e baixar o modelo

`llama3.2:3b` é o padrão, e não é preferência: o `1b` **foi medido e reprovado**
neste trabalho (10/26 na célula seguinte, errando para o lado do robô). Se você
trocar de modelo, rode a célula 3 antes de confiar.

O 3B em CPU também funciona, só devagar — a GPU do Colab existe aqui para a
medição ser rápida, não para o modelo caber.

In [ ]:
import os, shutil, time, subprocess, requests

MODELO = "llama3.2:3b"  # troque para medir: llama3.2:1b, qwen2.5:3b, gemma2:2b
OLLAMA = "http://127.0.0.1:11434"

# Mesma armadilha da célula anterior: binário ausente levanta exceção, não
# código de erro. Se a instalação falhou, o erro tem que dizer isso.
if not shutil.which("ollama"):
    raise SystemExit("O Ollama não está instalado — rode a célula 1 primeiro.")

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
# Mantém o modelo na GPU entre mensagens. Sem isto, cada pausa na conversa paga
# a carga de novo — o mesmo motivo do keep_alive em lib/chat-bot.ts.
os.environ["OLLAMA_KEEP_ALIVE"] = "60m"

servidor = subprocess.Popen(["ollama", "serve"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    if servidor.poll() is not None:
        raise SystemExit("o `ollama serve` morreu ao subir — rode a célula 1 de novo")
    try:
        requests.get(f"{OLLAMA}/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("o Ollama não respondeu em 60s")

print("Ollama de pé. Baixando", MODELO, "— alguns minutos na primeira vez.")
puxar = subprocess.run(["ollama", "pull", MODELO], capture_output=True, text=True)

# Conferir a LISTA, não o código de saída: é o que prova que o modelo está
# mesmo disponível para a próxima célula. Nome de modelo errado é o engano mais
# fácil aqui, e ele falha tarde — só quando a medição não encontra o modelo.
baixados = [m["name"] for m in requests.get(f"{OLLAMA}/api/tags").json()["models"]]
if not any(m == MODELO or m.startswith(f"{MODELO}:") for m in baixados):
    raise SystemExit(
        f"o modelo {MODELO} não ficou disponível.\n"
        f"{(puxar.stderr or puxar.stdout).strip()[-300:]}\n"
        f"baixados agora: {baixados or 'nenhum'}"
    )
print("pronto:", MODELO)

## 3. Medir antes de confiar

O modelo **não escreve** o que o devedor lê (decisão 32): ele devolve um rótulo
de intenção, e o texto sai de molde no inventário. Então o que importa medir não
é se ele escreve bonito — é se ele **classifica certo**.

A célula roda 26 falas de cobrança de verdade e conta duas coisas:

- **acertos**: quantos rótulos bateram com o esperado;
- **erros perigosos**: quantas vezes ele mandou para o robô algo que era de
  gente. Esses são os únicos que causam dano — errar para o lado da operadora
  custa atenção dela, errar para o lado do robô custa uma resposta indevida.

Serve para escolher modelo com número na mão. Referência medida em 12/08/2026:
`llama3.2:3b` fez **26/26 com zero erros perigosos**; `llama3.2:1b` fez **10/26**
e o erro dele tendia a `saudacao`, que é rótulo que o robô atende — por isso o
1B não serve para este trabalho.

O teto do inventário continua sendo **45s** por mensagem.

In [ ]:
import json, time, requests

# O prompt é o do inventário (lib/chat-intencao.ts). Colado aqui porque o
# notebook não enxerga o repositório — se você mexer lá, traga a mudança.
PROMPT = '''Você classifica a intenção de mensagens recebidas no WhatsApp de uma empresa de cobrança.

Responda SÓ com JSON: {"intencao":"<rótulo>","parcelas":<número ou null>}

Use EXATAMENTE um destes rótulos:
saudacao - cumprimento, "oi", "bom dia"
identificar - mandou CPF, data de nascimento ou nome para se identificar
consultar_saldo - quer saber quanto deve, qual o valor, qual a dívida
quer_negociar - quer parcelar, desconto, acordo, "como faço para pagar"
aceita - concorda com o que foi oferecido
recusa - não concorda, acha caro, não pode pagar
quer_boleto - pede boleto, código de barras, pix, segunda via
sobre_empresa - pergunta quem é a empresa ou o que ela faz
despedida - agradece ou se despede, "obrigado", "tchau"
ja_pagou - afirma que já pagou
contesta - diz que a dívida não é dele, não reconhece, fala em golpe
juridico - cita advogado, Procon, processo, justiça
outro - qualquer outra coisa

Na dúvida entre dois rótulos, responda "outro".

Exemplos:
"bom dia" -> {"intencao":"saudacao","parcelas":null}
"quanto eu devo" -> {"intencao":"consultar_saldo","parcelas":null}
"da pra parcelar em 6x" -> {"intencao":"quer_negociar","parcelas":6}
"pode ser" -> {"intencao":"aceita","parcelas":null}
"ta caro demais" -> {"intencao":"recusa","parcelas":null}
"manda o boleto" -> {"intencao":"quer_boleto","parcelas":null}
"ja paguei isso" -> {"intencao":"ja_pagou","parcelas":null}
"essa divida nao e minha" -> {"intencao":"contesta","parcelas":null}
"vou chamar meu advogado" -> {"intencao":"juridico","parcelas":null}
"o que voces fazem" -> {"intencao":"sobre_empresa","parcelas":null}
"obrigado, tchau" -> {"intencao":"despedida","parcelas":null}
"quanto custa uma pizza" -> {"intencao":"outro","parcelas":null}

"parcelas" só quando a pessoa disser um número de vezes. Senão null.'''

# Falas como gente escreve no WhatsApp: sem acento, com erro, curtas.
CASOS = [
    ("oi", "saudacao"), ("bom dia", "saudacao"), ("opa, tudo certo?", "saudacao"),
    ("quanto eu devo", "consultar_saldo"), ("qual o valor da minha divida?", "consultar_saldo"),
    ("me manda o valor", "consultar_saldo"),
    ("da pra parcelar?", "quer_negociar"), ("tem desconto?", "quer_negociar"),
    ("quero negociar isso", "quer_negociar"), ("como faco pra pagar", "quer_negociar"),
    ("pode ser", "aceita"), ("fechado, aceito", "aceita"),
    ("ta muito caro", "recusa"), ("nao consigo pagar isso", "recusa"),
    ("manda o boleto", "quer_boleto"), ("tem pix?", "quer_boleto"),
    ("o que voces fazem", "sobre_empresa"), ("que empresa e essa?", "sobre_empresa"),
    ("ja paguei isso", "ja_pagou"), ("paguei mes passado", "ja_pagou"),
    ("essa divida nao e minha", "contesta"), ("nao reconheco essa cobranca", "contesta"),
    ("vou falar com meu advogado", "juridico"), ("vou no procon", "juridico"),
    ("obrigado", "despedida"), ("quanto custa uma pizza", "outro"),
]

# Rótulos que o ROBÔ atende sozinho. Errar de "gente" para cá é o erro que custa.
DO_ROBO = {"saudacao", "consultar_saldo", "quer_negociar", "despedida",
           "identificar", "aceita", "recusa", "sobre_empresa"}

acertos, perigosos, erros, tempos = 0, 0, [], []
for fala, esperado in CASOS:
    t0 = time.time()
    try:
        r = requests.post(f"{OLLAMA}/api/chat", timeout=120, json={
            "model": MODELO,
            "messages": [{"role": "system", "content": PROMPT},
                         {"role": "user", "content": fala}],
            "stream": False, "format": "json", "keep_alive": "60m",
            "options": {"temperature": 0, "num_predict": 40},
        })
        veio = str(json.loads(r.json()["message"]["content"]).get("intencao", "")).strip()
    except Exception as e:
        veio = f"<{type(e).__name__}>"
    tempos.append(time.time() - t0)

    if veio == esperado:
        acertos += 1
    else:
        caro = esperado not in DO_ROBO and veio in DO_ROBO
        perigosos += caro
        erros.append((("!!" if caro else "  "), fala, esperado, veio))

print(f"{MODELO}:  {acertos}/{len(CASOS)} certos   {perigosos} erro(s) perigoso(s)")
print(f"tempo: média {sum(tempos)/len(tempos):.1f}s   pior {max(tempos):.1f}s   (teto: 45s)")
if erros:
    print("\nonde errou (!! = mandou para o robô algo que era de gente):")
    for marca, f, e, v in erros:
        print(f"  {marca} {f!r:<32} esperado {e:<16} veio {v}")

print()
if perigosos == 0 and acertos >= len(CASOS) * 0.9 and max(tempos) < 45:
    print("SERVE: classifica bem, erra para o lado seguro e cabe no teto.")
elif perigosos > 0:
    print("NÃO SERVE: mandou para o robô assunto que era de gente.")
elif max(tempos) >= 45:
    print("NÃO SERVE: acima do teto — as conversas cairiam na fila por tempo.")
else:
    print("FRACO: erra muito. Tente um modelo maior.")

## 4. Abrir a porta para o inventário

O Ollama **não tem autenticação nenhuma**. Num túnel público isso seria um
modelo aberto para quem achasse o endereço, e o endereço não é secreto.

Então quem atende o túnel não é o Ollama: é um proxy de vinte linhas que exige
um `Bearer` e só deixa passar **duas rotas** — conversar e listar modelos. Sem
isso, um `DELETE /api/delete` da internet apagaria o modelo no meio do
atendimento.

In [ ]:
import hmac, queue, re, secrets, shutil, socket, subprocess, sys, threading, time, requests

try:
    from flask import Flask, request, Response
except ImportError:
    # Python puro em vez de `!pip`: magic indentado dentro de `except` depende
    # do IPython transformar a linha, e isto aqui precisa funcionar sempre.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flask"], check=True)
    from flask import Flask, request, Response

TOKEN = secrets.token_urlsafe(32)

# Só o que o inventário usa. Lista de permissão, não de bloqueio: rota nova do
# Ollama nasce fechada em vez de nascer exposta.
LIBERADAS = {("POST", "/api/chat"), ("GET", "/api/tags")}

app = Flask(__name__)

@app.route("/<path:caminho>", methods=["GET", "POST"])
def repassar(caminho):
    rota = "/" + caminho
    if (request.method, rota) not in LIBERADAS:
        return Response('{"error":"rota fechada"}', 403, mimetype="application/json")

    # compare_digest: comparação de segredo não vaza o tamanho do acerto.
    enviado = request.headers.get("Authorization", "")
    if not hmac.compare_digest(enviado, f"Bearer {TOKEN}"):
        return Response('{"error":"token inválido"}', 401, mimetype="application/json")

    resp = requests.request(request.method, OLLAMA + rota,
                            data=request.get_data(), timeout=180,
                            headers={"content-type": "application/json"})
    return Response(resp.content, resp.status_code, mimetype="application/json")

# Porta escolhida na hora, e não fixa em 8000: reexecutar esta célula com o
# Flask anterior ainda de pé daria "address already in use", o servidor novo não
# subiria e o túnel apontaria para o antigo — que tem OUTRO token. O sintoma
# seria 401 no inventário, sem nada errado à vista.
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    PORTA = s.getsockname()[1]

threading.Thread(
    target=lambda: app.run(host="127.0.0.1", port=PORTA, threaded=True),
    daemon=True,
).start()

# Espera o servidor atender de fato. `time.sleep` fixo é aposta: em máquina
# carregada o túnel subiria antes do proxy existir.
for _ in range(50):
    try:
        requests.get(f"http://127.0.0.1:{PORTA}/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(0.2)
else:
    raise SystemExit("o proxy não subiu — rode a célula de novo")

# Túnel do Cloudflare: endereço público sem conta e sem cadastro. Ele MUDA a
# cada execução — é a fricção principal deste caminho.
if not shutil.which("cloudflared"):
    subprocess.run(
        ["curl", "-fL", "--retry", "5", "--retry-delay", "2", "--retry-all-errors",
         "-o", "/tmp/cf.deb",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/"
         "cloudflared-linux-amd64.deb"],
        check=True,
    )
    subprocess.run(["dpkg", "-i", "/tmp/cf.deb"], capture_output=True)

if not shutil.which("cloudflared"):
    raise SystemExit("o cloudflared não instalou — rode a célula de novo")

tunel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORTA}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# Ler o stdout do túnel direto no `for` penduraria a célula PARA SEMPRE se o
# cloudflared subisse mudo (rede bloqueada, serviço fora do ar). Uma thread
# despeja as linhas numa fila e aqui se espera com prazo: o notebook precisa
# poder desistir e dizer o motivo.
linhas = queue.Queue()
threading.Thread(target=lambda: [linhas.put(l) for l in tunel.stdout],
                 daemon=True).start()

url, prazo = None, time.time() + 60
while time.time() < prazo:
    try:
        achou = re.search(r"https://[-\w]+\.trycloudflare\.com",
                          linhas.get(timeout=5))
    except queue.Empty:
        if tunel.poll() is not None:
            raise SystemExit("o cloudflared morreu ao subir — rode a célula de novo")
        continue
    if achou:
        url = achou.group(0)
        break

if not url:
    tunel.terminate()
    raise SystemExit("o túnel não anunciou endereço em 60s — rode a célula de novo")

# O bloco do .env sai ANTES da conferência, e isso é a lição de uma vez que
# doeu: a checagem estourou (o DNS do túnel recém-criado ainda não resolvia
# dentro do Colab), levou a célula junto, e o endereço — o produto desta célula,
# que já existia — nunca chegou à tela. Verificação que destrói o que ela
# deveria validar é pior que não verificar.
print("\n" + "=" * 74)
print("Cole no .env do inventário e rode:  docker compose up -d")
print("=" * 74)
print(f'OLLAMA_URL="{url}"')
print(f'OLLAMA_MODELO="{MODELO}"')
print(f'OLLAMA_TOKEN="{TOKEN}"')
print("=" * 74)
print("Este endereço morre quando a sessão do Colab cair. Quando isso acontecer,")
print("rode esta célula de novo e troque as três linhas — ou apague OLLAMA_URL,")
print("que o atendimento volta a cair na fila da operadora sem quebrar nada.")

# Agora sim, a conferência — com paciência. Um túnel novo leva alguns segundos
# para o nome resolver, e o resolvedor do Colab não é o do seu escritório: não
# alcançar daqui NÃO quer dizer que não funciona lá.
print("\nconferindo o túnel", end="", flush=True)
for _ in range(12):
    try:
        com = requests.get(f"{url}/api/tags", timeout=10,
                           headers={"Authorization": f"Bearer {TOKEN}"}).status_code
        sem = requests.get(f"{url}/api/tags", timeout=10).status_code
        print(f"\n  com token: {com} (esperado 200)")
        print(f"  sem token: {sem} (esperado 401 — a porta está fechada)")
        break
    except requests.RequestException:
        print(".", end="", flush=True)
        time.sleep(5)
else:
    print("\n  não alcancei o endereço a partir do Colab em 60s.")
    print("  Quase sempre é o DNS do túnel ainda propagando; o endereço acima")
    print("  continua valendo. Confira do servidor do inventário:")
    print(f'    curl -H "Authorization: Bearer {TOKEN}" {url}/api/tags')

## 5. Segurar a sessão viva

Deixe esta célula rodando **e a aba aberta**. Ela também é o seu monitor: mostra
quantas mensagens o inventário mandou e avisa quando o modelo cair.

Para desligar tudo: interrompa a célula e feche a aba. No inventário, apague
`OLLAMA_URL` do `.env` — o atendimento volta inteiro para a fila da operadora,
sem quebrar nada.

In [ ]:
import time, requests
from datetime import datetime, timedelta, timezone
from IPython.display import clear_output

# Rodar esta célula sozinha (depois de a sessão cair, por exemplo) não monitora
# coisa nenhuma: as variáveis morreram junto. Melhor dizer isso do que estourar
# um NameError seco.
if "OLLAMA" not in dir() or "url" not in dir():
    raise SystemExit("rode as células 1 a 4 primeiro — a sessão perdeu o estado")

BRASIL = timezone(timedelta(hours=-3))
inicio = time.time()

while True:
    try:
        vivo = requests.get(f"{OLLAMA}/api/tags", timeout=5).status_code == 200
    except Exception:
        vivo = False

    horas = (time.time() - inicio) / 3600
    agora = datetime.now(BRASIL).strftime("%H:%M")
    estado = "ok" if vivo else "MODELO FORA DO AR — o inventário está escalando tudo"

    # `clear_output` e não "\r": o Colab não reescreve a linha com carriage
    # return, ele empilha. Em 12h seriam 720 linhas engordando o .ipynb e a aba.
    clear_output(wait=True)
    linhas = [f"{agora}   de pé há {horas:4.1f}h   {estado}", ""]
    # O endereço e o segredo repetidos aqui de propósito: eles saem da tela
    # assim que a célula 4 rola para cima, e são o que você precisa ter à mão
    # para consertar o .env quando alguma coisa der errado.
    linhas += ["Para o .env do inventário:",
               f'OLLAMA_URL="{url}"', f'OLLAMA_MODELO="{MODELO}"',
               f'OLLAMA_TOKEN="{TOKEN}"']
    if horas > 11.5:
        linhas += ["", "perto do teto de 12h do Colab: a sessão vai cair em breve."]
    print("\n".join(linhas))

    time.sleep(60)